# 01 Data audit and EDA
Point-in-time audit, leakage checks and split justification for the no-show triage project.

Make the project folder importable.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

Import libraries and project modules.

In [2]:
from src import config, data, evaluate
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")


pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

Load the raw file and show its shape and column types.

In [3]:
raw = data.load_raw()
print(raw.shape)
display(raw.dtypes.to_frame("dtype"))
display(raw.head())

(110527, 14)


,dtype
patient_id,float64
appointment_id,int64
gender,object
scheduled_day,object
appointment_day,object
age,int64
neighbourhood,object
scholarship,int64
hipertension,int64
diabetes,int64


,patient_id,appointment_id,gender,scheduled_day,appointment_day,age,neighbourhood,scholarship,hipertension,diabetes,alcoholism,handcap,sms_received,no_show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


Record the data hash, file size and row count for data/README.md.

In [4]:
md5 = data.file_md5()
size_mb = config.DATA_RAW.stat().st_size / 1e6
print("file:", config.DATA_RAW.name)
print("md5:", md5)
print("size MB:", round(size_mb, 2))
print("rows:", len(raw), "columns:", raw.shape[1])

file: KaggleV2-May-2016.csv
md5: cc55525de6e2b615aeba50095e8aaa95
size MB: 10.74
rows: 110527 columns: 14


Show the point-in-time audit of every column.

In [5]:
audit = pd.DataFrame(config.COLUMN_AUDIT)
with pd.option_context("display.max_colwidth", 80):
    display(audit)

,column,written_when,available_at_decision_time,role,note
0,PatientId,At patient registration,Yes,Key only; used to build prior-appointment history,Never a feature itself
1,AppointmentID,At booking,Yes,Row key,Checked for duplicates
2,Gender,At registration,Yes,Slice analysis only,Excluded as input by charter
3,ScheduledDay,At booking,Only if booked before 17:00 the day before,"Feature (lead time, booking hour, weekday)",Rows booked later are outside the scored population
4,AppointmentDay,At booking,Yes,Feature (weekday) and split key,"Date only, no clock time in the file"
5,Age,At registration or booking,Yes,Feature,Invalid values removed
6,Neighbourhood,At registration,Yes,Slice analysis only,Excluded as input by charter
7,Scholarship,Enrolment flag; timestamp not in the file,Assumed yes (VERIFY),Feature,Assumption stated in report
8,Hipertension,Patient record flag; timestamp not in the file,Assumed yes (VERIFY),Feature,Assumption stated in report
9,Diabetes,Patient record flag; timestamp not in the file,Assumed yes (VERIFY),Feature,Assumption stated in report


Count missing values per column in the raw file.

In [6]:
display(raw.isna().sum().to_frame("missing"))

,missing
patient_id,0
appointment_id,0
gender,0
scheduled_day,0
appointment_day,0
age,0
neighbourhood,0
scholarship,0
hipertension,0
diabetes,0


Check identifier quality: unique patients, unique appointments and truncated-looking patient ids.

In [7]:
pid = pd.to_numeric(raw["patient_id"]).round().astype("int64")
print("rows:", len(raw))
print("unique appointment ids:", raw["appointment_id"].nunique())
print("unique patients:", pid.nunique())
print("share of patient ids that are multiples of 1,000,000:",
      round(float((pid % 10**6 == 0).mean()), 4))

rows: 110527
unique appointment ids: 110527
unique patients: 62299
share of patient ids that are multiples of 1,000,000: 0.0


Inspect invalid ages before cleaning.

In [8]:
print(raw["age"].describe())
print("negative ages:", int((raw["age"] < 0).sum()))
print("ages above", config.AGE_MAX, ":", int(
    (raw["age"] > config.AGE_MAX).sum()))

count    110527.000000
mean         37.088874
std          23.110205
min          -1.000000
25%          18.000000
50%          37.000000
75%          55.000000
max         115.000000
Name: age, dtype: float64
negative ages: 1
ages above 110 : 5


Clean the data and report every row removed and why.

In [9]:
cleaned, clean_report = data.clean(raw)
display(pd.Series(clean_report).to_frame("rows"))

,rows
rows_raw,110527
rows_duplicate_appointment_id,0
rows_invalid_age,6
rows_booked_after_appointment_date,5
rows_clean,110521


Build the model table with history features, the 17:00 eligibility rule and the time split.

In [10]:
table, report = data.build_model_table()
report["data_md5"] = md5
display(pd.Series(report).to_frame("value"))

,value
rows_raw,110527
rows_duplicate_appointment_id,0
rows_invalid_age,6
rows_booked_after_appointment_date,5
rows_clean,110521
rows_not_booked_by_decision_time,38874
no_show_rate_not_booked_by_decision_time,0.04795
rows_model_table,71647
rows_train,60126
rows_test,11521


Compare appointments booked after the 17:00 decision point with the scored population.

In [11]:
train = table[table["split"] == "train"].copy()
share_out = report["rows_not_booked_by_decision_time"] / report["rows_clean"]
print("share of clean rows booked after the decision point:", round(share_out, 3))
print("their no-show rate:",
      round(report["no_show_rate_not_booked_by_decision_time"], 3))
print("no-show rate of the scored population (train split):",
      round(float(train["no_show"].mean()), 3))

share of clean rows booked after the decision point: 0.352
their no-show rate: 0.048
no-show rate of the scored population (train split): 0.29


Show class balance on the training split.

In [12]:
display(train["no_show"].value_counts(
    normalize=True).rename("share").to_frame())

,share
no_show,
0,0.709876
1,0.290124


Show split sizes and date ranges without looking at test labels.

In [13]:
display(table.groupby("split").agg(rows=("no_show", "size"), first_day=(
    "appointment_day", "min"), last_day=("appointment_day", "max")))

,rows,first_day,last_day
split,,,
test,11521,2016-06-03,2016-06-08
train,60126,2016-04-29,2016-06-02


Plot the age and lead-time distributions.

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(table["age"], bins=40)
axes[0].set_title("Age")
axes[1].hist(table["lead_days"].clip(upper=90), bins=45)
axes[1].set_title("Lead time in days (clipped at 90)")
fig.tight_layout()
plt.show()

Plot no-show rate by lead-time band on the training split.

In [15]:
banded = evaluate.add_slice_columns(train)
order = ["1 day", "2-7 days", "8-14 days", "15+ days"]
by_lead = banded.groupby("lead_band")["no_show"].agg(
    ["mean", "size"]).reindex(order)
display(by_lead)
by_lead["mean"].plot.bar(title="No-show rate by lead time (train)", rot=0)
plt.show()

,mean,size
lead_band,,
1 day,0.216054,4161
2-7 days,0.250881,22413
8-14 days,0.305654,10754
15+ days,0.334898,22798


Plot no-show rate by age band and appointment weekday on the training split.

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
age_order = ["0-17", "18-34", "35-49", "50-64", "65+"]
banded.groupby("age_band")["no_show"].mean().reindex(
    age_order).plot.bar(ax=axes[0], title="By age band (train)", rot=0)
banded.groupby("appointment_weekday")["no_show"].mean().plot.bar(
    ax=axes[1], title="By appointment weekday, 0 = Monday (train)", rot=0)
fig.tight_layout()
plt.show()

Measure how much prior-appointment history exists and whether it relates to no-shows.

In [17]:
print("share of appointments with any prior history:",
      round(float((train["prior_appts"] > 0).mean()), 3))
display(train.groupby("has_history")["no_show"].agg(["mean", "size"]))
display(train["prior_appts"].describe().to_frame("prior_appts"))

share of appointments with any prior history: 0.355


,mean,size
has_history,,
0,0.286628,38754
1,0.296463,21372


,prior_appts
count,60126.000000
mean,0.768420
std,1.797728
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,65.000000


Scan each candidate column alone for suspiciously strong prediction (leakage check).

In [18]:
from sklearn.metrics import roc_auc_score

candidates = config.NUMERIC_FEATURES + ["sms_received", "handcap"]
rows = []
for col in candidates:
    values = train[col].astype(float)
    values = values.fillna(values.median())
    auc = roc_auc_score(train["no_show"], values)
    rows.append({"column": col, "single_column_auc": round(auc, 4),
                "distance_from_chance": round(abs(auc - 0.5), 4)})
scan = pd.DataFrame(rows).sort_values(
    "distance_from_chance", ascending=False).reset_index(drop=True)
display(scan)
print("columns with single-column AUC above 0.80 or below 0.20:",
      scan.loc[scan["distance_from_chance"] > 0.30, "column"].tolist())

,column,single_column_auc,distance_from_chance
0,age,0.4340,0.0660
1,lead_days,0.5568,0.0568
2,lead_hours,0.5559,0.0559
3,hipertension,0.4754,0.0246
4,scheduled_hour,0.5243,0.0243
5,prior_noshow_rate,0.5178,0.0178
6,prior_noshows,0.5175,0.0175
7,scholarship,0.5155,0.0155
8,sms_received,0.4885,0.0115
9,appointment_weekday,0.4911,0.0089


columns with single-column AUC above 0.80 or below 0.20: []


Show why sms_received is excluded: how it moves with lead time next to the no-show rate.

In [19]:
display(banded.groupby("lead_band")[
        ["sms_received", "no_show"]].mean().reindex(order))

,sms_received,no_show
lead_band,,
1 day,0.000000,0.216054
2-7 days,0.387409,0.250881
8-14 days,0.552632,0.305654
15+ days,0.564611,0.334898


Save the processed model table and cleaning report.

In [20]:
data.save_processed(table, report)
print(sorted(p.name for p in config.DATA_PROCESSED_DIR.iterdir()))

['appointments_model_table.csv', 'cleaning_report.json']


## Findings
Write 4 to 6 sentences here after running: the leakage verdict from the scan, the sms_received decision, why the time split fits deployment, and the data-quality issues removed.